# Iran Provincial Multidimensional Composite Index — V2.1

This notebook is a portfolio-facing walkthrough of the **frozen final outputs**. It explains the analytical pipeline and reproduces key views from the CSV outputs rather than creating a second scoring engine.

## 1. Research question

**How can heterogeneous provincial indicators be transformed into a transparent, reproducible and robustness-tested multidimensional composite index?**

The unit of analysis is province × logical series × year.

## 2. Model architecture

Raw workbook → standardization → duplicate/conflict audit → registry → direction/unit/transform → population standardization → normalization → recency weighting → temporal aggregation → Series → Subindicator → Category → Overall → robustness → validation.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'outputs').exists():
    ROOT = Path('..').resolve()

composite = pd.read_csv(ROOT / 'outputs' / 'provincial_composite_index.csv')
category = pd.read_csv(ROOT / 'outputs' / 'category_scores.csv')
mc = pd.read_csv(ROOT / 'outputs' / 'monte_carlo_local.csv')
registry = pd.read_csv(ROOT / 'registry' / 'indicator_registry_final.csv')

print('Provinces:', composite['province'].nunique())
print('Registry series:', registry['series_id'].nunique())
print('Active series:', (registry['inclusion_status'] == 'CONDITIONAL').sum())
print('Categories:', category['category'].nunique())


## 3. Final provincial ranking

The ranking is a comparative ordering of the frozen composite score. It should be read together with the model assumptions and robustness diagnostics.

In [ ]:
ranked = composite.sort_values('overall_score', ascending=True)

fig, ax = plt.subplots(figsize=(10, 12))
ax.barh(ranked['province'], ranked['overall_score'])
ax.set_xlabel('Composite score')
ax.set_title('Final provincial composite ranking')
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()


## 4. Domain profile

The five domain scores are exposed separately so the final score is explainable rather than a black box.

In [ ]:
pivot = category.pivot(index='province', columns='category', values='category_score')

fig, ax = plt.subplots(figsize=(10, 12))
im = ax.imshow(pivot.values, aspect='auto')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
ax.set_title('Province × domain score')
fig.colorbar(im, ax=ax, label='Domain score')
plt.tight_layout()
plt.show()


## 5. Local Monte Carlo stability

The local simulation perturbs weights around the baseline and summarizes the resulting rank interval. These intervals are **model-stability diagnostics**, not real-world probabilities.

In [ ]:
mc_sorted = mc.sort_values('median_rank', ascending=False)

fig, ax = plt.subplots(figsize=(10, 12))
y = np.arange(len(mc_sorted))
ax.hlines(y, mc_sorted['rank_p05'], mc_sorted['rank_p95'])
ax.scatter(mc_sorted['median_rank'], y)
ax.set_yticks(y)
ax.set_yticklabels(mc_sorted['province'])
ax.invert_yaxis()
ax.invert_xaxis()
ax.set_xlabel('Rank (P05–P95 interval)')
ax.set_title('Local Monte Carlo rank stability')
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()


## 6. Population standardization

Count-based indicators are transformed only where population normalization is substantively meaningful. School/class counts use same-year Population6Plus when available; other selected counts use same-year total population, then a nearby total-population observation within a maximum two-year gap when needed. Rates, percentages, ratios, indexes and existing per-capita measures are not divided by population again.

In [ ]:
transformed = pd.read_csv(ROOT / 'data' / 'processed' / 'transformed_observations.csv')
cols = ['series_id','transform','denominator_year','denominator_gap','denominator_type']
print(transformed[cols].drop_duplicates().head(20).to_string(index=False))


## 7. Robustness summary

Key final diagnostics are documented in `docs/robustness.md` and the corresponding CSV outputs. Temporal specifications are highly stable; some domain-weight changes are materially influential. This is why the baseline is presented with sensitivity results rather than as a uniquely correct ranking.

## 8. Limitations

The final model is portfolio-ready but not presented as fully externally validated. Active series remain conditional because complete row-level provenance is not recoverable from the supplied workbook metadata. SHDI provides convergent rather than independent validation.